# MASIVE-ALS — Cribado MASIVO TDP43_v2 (Notebook Maestro)

Descarga TODA la libreria desde GitHub (311 archivos tar.gz, ~105K ligandos).
Usa Vina-GPU-2.1 con checkpoint. 3 semillas (42, 2026, 777).
Si se relanza, continua donde quedo (checkpoint en hechos_master.txt).

**Receptor:** TDP43_v2 (4BS2, bolsillo nuevo validado 26/08)
**Blanco:** Interfaz RRM1-RRM2 Arg151-Asp247
**Caja:** centro (24.23, 16.89, -15.87), tamano 26 A

1. Activar GPU en el panel derecho
2. Run All

In [ ]:
!nvidia-smi
print('GPU OK')

In [ ]:
# CELDA 2: Obtener binario Vina-GPU
import os, glob, subprocess, urllib.request, tarfile, shutil

r = subprocess.run(['apt-get', 'update', '-qq'], capture_output=True, text=True)
r = subprocess.run(['apt-get', 'install', '-y', '-qq', 'libboost-program-options1.74.0'], capture_output=True, text=True)
print('boost rc=%d' % r.returncode)
if r.returncode != 0:
    subprocess.run(['pip', 'install', '-q', 'libboost'], capture_output=True, text=True)

VINA_GPU_URL = 'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main/vinagpu_linux.tar.gz'
BIN_DIR = '/kaggle/working/vinagpu_linux'
os.makedirs(BIN_DIR, exist_ok=True)

def buscar_binario(base):
    if not base or not os.path.exists(base):
        return None
    hits = glob.glob(base + '/**/AutoDock-Vina-GPU-2-1', recursive=True)
    return hits[0] if hits else None

VINA_GPU_BIN = buscar_binario(BIN_DIR)
if not VINA_GPU_BIN:
    pkg = '/tmp/vinagpu_linux.tar.gz'
    urllib.request.urlretrieve(VINA_GPU_URL, pkg)
    with tarfile.open(pkg) as t:
        t.extractall(BIN_DIR)
    VINA_GPU_BIN = buscar_binario(BIN_DIR)
if not VINA_GPU_BIN:
    raise SystemExit('ERROR: no se pudo obtener Vina-GPU')
BIN_DIR = os.path.dirname(VINA_GPU_BIN)
os.chmod(VINA_GPU_BIN, 0o755)
print('Vina-GPU listo:', VINA_GPU_BIN)

In [ ]:
# CELDA 3: Descargar receptor TDP43_v2
WORK = '/kaggle/working/masive_als'
for sub in ['receptores', 'ligandos', 'resultados', 'checkpoint']:
    os.makedirs(WORK + '/' + sub, exist_ok=True)

# Receptor desde GitHub (masive-als-data)
GH_DATA = 'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main'
receptor_url = GH_DATA + '/receptores_tdp43v2.tar'
receptor_tar = '/tmp/receptores_tdp43v2.tar'
if not os.path.exists(WORK + '/receptores/TDP43_v2.pdbqt'):
    print('Descargando receptor...')
    urllib.request.urlretrieve(receptor_url, receptor_tar)
    with tarfile.open(receptor_tar) as t:
        t.extractall(WORK)
else:
    print('Receptor ya existe')

print('Receptor TDP43_v2:', os.path.exists(WORK + '/receptores/TDP43_v2.pdbqt'))

In [ ]:
# CELDA 4: Obtener lista de archivos tar.gz desde GitHub API
import json

GH_API = 'https://api.github.com/repos/fredy30-Rojas/masive-als-data/contents/'
req = urllib.request.Request(GH_API)
with urllib.request.urlopen(req) as resp:
    contents = json.loads(resp.read())

# Filtrar solo tar.gz de ligandos (no receptores, no binarios)
SKIP = {'vinagpu_linux.tar.gz', 'colab_receptores_plano.tar.gz', 'receptores_tdp43v2.tar',
        'receptores_tdp43v2.tar.gz', 'qvina_kaggle.tar.gz'}
ligand_tars = []
for item in contents:
    name = item['name']
    if name.endswith('.tar.gz') and name not in SKIP and 'receptor' not in name.lower():
        ligand_tars.append({'name': name, 'url': item['download_url'], 'size': item['size']})

ligand_tars.sort(key=lambda x: x['name'])
total_size_mb = sum(t['size'] for t in ligand_tars) / 1024 / 1024
print(f'Archivos de ligandos: {len(ligand_tars)}')
print(f'Tamano total: {total_size_mb:.1f} MB')
print(f'Primeros 5: {[t["name"] for t in ligand_tars[:5]]}')

In [ ]:
# CELDA 5: Descargar TODOS los ligandos y extraerlos
import time

LIG_DIR = WORK + '/ligandos'
os.makedirs(LIG_DIR, exist_ok=True)

# Checkpoint: que archivos ya descargamos
descargados_file = WORK + '/checkpoint/descargados.txt'
descargados = set()
if os.path.exists(descargados_file):
    descargados = set(l.strip() for l in open(descargados_file) if l.strip())

print(f'Ya descargados: {len(descargados)} archivos')
print(f'Por descargar: {len(ligand_tars) - len(descargados)} archivos')

t0 = time.time()
n_new = 0
for i, item in enumerate(ligand_tars):
    if item['name'] in descargados:
        continue
    try:
        tar_path = '/tmp/_lig/' + item['name']
        os.makedirs('/tmp/_lig', exist_ok=True)
        urllib.request.urlretrieve(item['url'], tar_path)
        with tarfile.open(tar_path) as t:
            for m in t.getmembers():
                if m.name.endswith('.pdbqt') and 'receptor' not in m.name.lower():
                    fname = os.path.basename(m.name)
                    dst = LIG_DIR + '/' + fname
                    if not os.path.exists(dst):
                        with t.extractfile(m) as src:
                            with open(dst, 'wb') as f:
                                f.write(src.read())
        os.remove(tar_path)
        descargados.add(item['name'])
        with open(descargados_file, 'a') as f:
            f.write(item['name'] + '\n')
        n_new += 1
    except Exception as ex:
        print(f'ERROR {item["name"]}: {str(ex)[:80]}')
    if n_new % 20 == 0 and n_new > 0:
        print(f'[{n_new} archivos nuevos, {len(os.listdir(LIG_DIR))} ligandos total] {time.time()-t0:.0f}s')

total_lig = len(os.listdir(LIG_DIR))
print(f'\nDescarga completada: {n_new} archivos nuevos')
print(f'Total ligandos: {total_lig}')
print(f'Tiempo: {(time.time()-t0)/60:.1f} min')

In [ ]:
# CELDA 6: Receptor TDP43_v2
RECEPTORES = {
    'TDP43_v2': {
        'archivo': WORK + '/receptores/TDP43_v2.pdbqt',
        'centro': [24.23, 16.89, -15.87],
        'tamano': [26, 26, 26]
    },
}
print('Receptor listo:', os.path.exists(RECEPTORES['TDP43_v2']['archivo']))

In [ ]:
# CELDA 7: Pipeline de docking masivo con Vina-GPU
import csv

OUT_DIR = WORK + '/resultados'
CSV = OUT_DIR + '/resultados_master_tdp43v2.csv'
CKPT = OUT_DIR + '/hechos_master.txt'

if not os.path.exists(CSV):
    with open(CSV, 'w', newline='') as f:
        csv.writer(f).writerow(['ligand', 'target', 'energy', 'seed', 'timestamp'])

hechos = set()
if os.path.exists(CKPT):
    hechos = set(l.strip() for l in open(CKPT) if l.strip())
print('Hechos antes:', len(hechos))

ligandos = sorted(glob.glob(LIG_DIR + '/*.pdbqt'))
print('Ligandos totales:', len(ligandos))
print('Por hacer:', len(ligandos) - len(hechos))

SEEDS = [42, 2026, 777]

def tiene_atomos(lig):
    try:
        with open(lig, errors='replace') as f:
            return any(l.startswith(('ATOM', 'HETATM')) for l in f)
    except:
        return False

def acoplar(lig, target, seed, thread=8000):
    info = RECEPTORES[target]
    cfg = '/tmp/cfg_%s_%d.txt' % (os.path.basename(lig).replace('.pdbqt','')[:25], seed)
    with open(cfg, 'w') as f:
        f.write('receptor = %s\n' % info['archivo'])
        f.write('ligand = %s\n' % lig)
        f.write('center_x = %s\n' % info['centro'][0])
        f.write('center_y = %s\n' % info['centro'][1])
        f.write('center_z = %s\n' % info['centro'][2])
        f.write('size_x = %s\n' % info['tamano'][0])
        f.write('size_y = %s\n' % info['tamano'][1])
        f.write('size_z = %s\n' % info['tamano'][2])
        f.write('num_modes = 3\n')
        f.write('seed = %d\n' % seed)
        f.write('thread = %d\n' % thread)
    try:
        r = subprocess.run([VINA_GPU_BIN, '--config', cfg], capture_output=True, text=True,
                           timeout=1800, cwd=BIN_DIR)
        out = (r.stdout or '') + (r.stderr or '')
        if r.returncode != 0:
            return None, 'rc=%d %s' % (r.returncode, out[-150:])
        for ln in out.splitlines():
            s = ln.split()
            if len(s) >= 2 and s[0] == '1':
                try:
                    return round(float(s[1]), 4), None
                except ValueError:
                    pass
        return None, 'sin afinidad'
    except Exception as ex:
        return None, str(ex)[:100]

t0 = time.time()
n_ok = 0
n_err = 0
n_skip = 0

for lig in ligandos:
    nombre = os.path.basename(lig).replace('.pdbqt', '')
    if nombre in hechos:
        n_skip += 1
        continue
    if not tiene_atomos(lig):
        hechos.add(nombre)
        n_err += 1
        continue
    for target in RECEPTORES:
        energias = []
        err = None
        for sd in SEEDS:
            e, er = acoplar(lig, target, sd)
            if e is not None:
                energias.append(e)
            else:
                err = er
        if energias:
            best = min(energias)
            best_seed = SEEDS[energias.index(best)]
            with open(CSV, 'a', newline='') as f:
                csv.writer(f).writerow([nombre, target, best, best_seed,
                                       time.strftime('%Y-%m-%d %H:%M:%S')])
            n_ok += 1
        else:
            n_err += 1
            print('ERROR', nombre, target, err)
    with open(CKPT, 'a') as f:
        f.write(nombre + '\n')
    hechos.add(nombre)
    if n_ok % 50 == 0 and n_ok > 0:
        elapsed = (time.time() - t0) / 60
        remaining = len(ligandos) - len(hechos)
        rate = n_ok / elapsed if elapsed > 0 else 0
        eta = remaining / rate if rate > 0 else 0
        print('[%d ok, %d err, %d skip] %.1f min | ETA: %.1f h' % (
            n_ok, n_err, n_skip, elapsed, eta/60), flush=True)

elapsed = (time.time() - t0) / 60
print()
print('=== CRIBADO MASTER COMPLETADO ===')
print('Acoplados:', n_ok, '| Errores:', n_err, '| Skip:', n_skip)
print('Tiempo total: %.1f min' % elapsed)
print('CSV:', CSV)

In [ ]:
# CELDA 8: Resumen y top hits
import csv

rows = list(csv.DictReader(open(CSV)))
print('Total resultados:', len(rows))

if rows:
    best = sorted(rows, key=lambda x: float(x['energy']))[:20]
    print()
    print('Top 20 hits contra TDP43_v2:')
    print('%-20s %s' % ('Ligando', 'Afinidad'))
    print('-' * 35)
    for r in best:
        print('%-20s %s' % (r['ligand'], r['energy']))

print()
print('Para descargar: resultados_master_tdp43v2.csv')